<h1 style="color:#56B4E9;">Week 9 — Day 1</h1>

<h2 style="color:#0072B2;">Sprint 4 Planning, Reproducibility & Serialization</h2>

**Final BinX project:** an educational platform that classifies posts, analyzes teaching feedback, and recommends posts and people.

> Today we plan the MVP and verify the complete train → save → load → predict path. The small model in this notebook is a **serialization smoke test**, not a trained production model or a reported benchmark.

<h2 style="color:#0072B2;">Learning Objectives</h2>

By the end of Day 1, we will:

1. Freeze the Sprint 4 goal and backlog.
2. Define the three AI systems and their contracts.
3. Confirm that post classification is multi-label.
4. Set reproducibility controls.
5. train a tiny smoke-test pipeline, serialize it, reload it, and reproduce the same prediction.
6. Separate completed evidence from future project work.

In [1]:
from pathlib import Path
from importlib.metadata import version
import json
import os
import platform
import random
import tempfile

import joblib
import numpy as np
import pandas as pd
import sklearn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

environment = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "joblib": version("joblib"),
    "seed": SEED,
}
pd.Series(environment, name="value").to_frame()

,value
python,3.12.3
numpy,2.5.1
pandas,2.3.3
scikit-learn,1.9.0
joblib,1.5.3
seed,42


<h2 style="color:#0072B2;">1.1 Sprint 4 Goal</h2>

<span style="color:#009E73;"><b>Sprint 4 Goal:</b></span>  
Build a reproducible MVP that exposes three separate AI responsibilities:

- propose educational topics for English posts and their images;
- analyze teaching-related aspects in written feedback while keeping star ratings separate;
- recommend eligible posts and people using transparent content matching.

<span style="color:#D55E00;"><b>Scope boundary:</b></span>  
The project contains **no payment, paid-content, discount, or financial-reward system**. Recommendation never grants access to restricted content; the backend remains responsible for authorization.

In [2]:
systems = pd.DataFrame([
    {"ID": "A", "System": "Post topic classification", "Input": "English title/body + optional image", "Output": "One or more topics + scores"},
    {"ID": "B", "System": "Teaching-feedback analysis", "Input": "Review text + separate 1–5 stars", "Output": "Teaching aspects + bounded sentiment signal"},
    {"ID": "C", "System": "Post/person recommendation", "Input": "Interests + eligible candidates", "Output": "Ranked IDs + safe reason codes"},
])
systems

,ID,System,Input,Output
0,A,Post topic classification,English title/body + optional image,One or more topics + scores
1,B,Teaching-feedback analysis,Review text + separate 1–5 stars,Teaching aspects + bounded sentiment signal
2,C,Post/person recommendation,Interests + eligible candidates,Ranked IDs + safe reason codes


<h2 style="color:#0072B2;">1.2 Sprint 4 Backlog</h2>

The official Week 9 deployment flow is adapted to this new project. A public claim is allowed only after a real dataset, held-out evaluation, and repeatable result exist.

In [3]:
backlog = pd.DataFrame([
    {"Day": 1, "Deliverable": "Scope, taxonomy, contracts, reproducibility and serialization smoke test", "Done when": "Reloaded pipeline reproduces the same output"},
    {"Day": 2, "Deliverable": "FastAPI service contracts and prediction endpoints", "Done when": "Valid and invalid requests are tested through /docs"},
    {"Day": 3, "Deliverable": "Streamlit dashboard connected to the service", "Done when": "A new user can obtain and understand an output"},
    {"Day": 4, "Deliverable": "Public deployment and environment reconciliation", "Done when": "Live and local outputs match on fixed examples"},
    {"Day": 5, "Deliverable": "Repository polish, tests, write-up and retrospective", "Done when": "Definition of Done checklist passes"},
])
backlog

,Day,Deliverable,Done when
0,1,"Scope, taxonomy, contracts, reproducibility an...",Reloaded pipeline reproduces the same output
1,2,FastAPI service contracts and prediction endpo...,Valid and invalid requests are tested through ...
2,3,Streamlit dashboard connected to the service,A new user can obtain and understand an output
3,4,Public deployment and environment reconciliation,Live and local outputs match on fixed examples
4,5,"Repository polish, tests, write-up and retrosp...",Definition of Done checklist passes


<h2 style="color:#0072B2;">1.3 Topic Taxonomy Decision</h2>

Post classification is **multi-label**, because one post may discuss AI, Python, and robotics together.

The eight labels below are the MVP draft.

<span style="color:#D55E00;"><b>Expansion rule:</b></span>  
Health, Business, Languages, Humanities, Education, and General Engineering remain expansion candidates until the team confirms scope and suitable data.

In [4]:
MVP_TOPICS = [
    "Programming/Web",
    "AI/Data",
    "Electronics/Embedded",
    "Robotics",
    "Cybersecurity",
    "Design",
    "Mathematics",
    "Natural Sciences",
]

assert len(MVP_TOPICS) == len(set(MVP_TOPICS)) == 8
pd.DataFrame({"MVP topic": MVP_TOPICS}, index=range(1, len(MVP_TOPICS) + 1))

,MVP topic
1,Programming/Web
2,AI/Data
3,Electronics/Embedded
4,Robotics
5,Cybersecurity
6,Design
7,Mathematics
8,Natural Sciences


<h2 style="color:#0072B2;">1.4 Data and Model Plan</h2>

<span style="color:#D55E00;"><b>Scientific honesty rule:</b></span>  
These datasets are **candidates, not adopted training corpora**. We must audit licence, label mapping, class balance, duplicates, and split leakage before reporting results.

In [5]:
model_plan = pd.DataFrame([
    {"System": "A — Text topics", "Candidate data": "EngineeringConcepts + audited Wikibooks labels", "First baseline": "TF-IDF + One-vs-Rest Logistic Regression", "Primary metrics": "Per-topic precision, recall, F1"},
    {"System": "A — Images", "Candidate data": "MMMU / ScienceQA exploration + project-style test set", "First baseline": "OCR → text classifier; evaluate CLIP separately", "Primary metrics": "Per-topic F1 + human-review rate"},
    {"System": "B — Feedback", "Candidate data": "EduRABSA; optional independent student-feedback set", "First baseline": "Teaching-aspect sentiment classifier", "Primary metrics": "Aspect F1 and sentiment macro-F1"},
    {"System": "C — Recommendations", "Candidate data": "Explicit interests + eligible content; MIND-small for rehearsal only", "First baseline": "TF-IDF vs sentence embeddings + cosine ranking", "Primary metrics": "Recall@K, NDCG@K, coverage"},
])
model_plan

,System,Candidate data,First baseline,Primary metrics
0,A — Text topics,EngineeringConcepts + audited Wikibooks labels,TF-IDF + One-vs-Rest Logistic Regression,"Per-topic precision, recall, F1"
1,A — Images,MMMU / ScienceQA exploration + project-style t...,OCR → text classifier; evaluate CLIP separately,Per-topic F1 + human-review rate
2,B — Feedback,EduRABSA; optional independent student-feedbac...,Teaching-aspect sentiment classifier,Aspect F1 and sentiment macro-F1
3,C — Recommendations,Explicit interests + eligible content; MIND-sm...,TF-IDF vs sentence embeddings + cosine ranking,"Recall@K, NDCG@K, coverage"


Useful starting points: [EngineeringConcepts](https://huggingface.co/datasets/kd13/EngineeringConcepts-Instruct-v1), [Wikibooks dumps](https://dumps.wikimedia.org/enwikibooks/20260301/), [MMMU](https://huggingface.co/datasets/MMMU/MMMU), [EduRABSA](https://github.com/yhua219/edurabsa_dataset_and_annotation_tool), and [MIND](https://msnews.github.io/).

<h2 style="color:#0072B2;">1.5 Input and Output Contracts</h2>

Stable contracts let the future notebook, FastAPI service, Streamlit UI, and backend agree on the same fields.

<span style="color:#D55E00;"><b>Score rule:</b></span>  
Scores rank candidates inside one model version; they are not truth or learning-quality percentages.

In [6]:
contracts = {
    "topic_request": {"post_id": "post_42", "title": "ESP32 robot", "body": "PID wall following", "image_ref": None},
    "topic_response": {"topics": [{"id": "Robotics", "score": 0.82}], "model_version": "topic-v1"},
    "feedback_request": {"review_id": "review_7", "text": "Clear explanation but few examples", "stars": 4},
    "feedback_response": {"aspects": [{"name": "clarity", "sentiment": "positive"}, {"name": "examples", "sentiment": "negative"}], "text_signal": 0.10, "stars_unchanged": 4},
    "recommend_request": {"user_id": "user_8", "surface": "posts", "declared_topics": ["Programming/Web"], "eligible_candidate_ids": ["post_42", "post_91"]},
    "recommend_response": {"items": [{"id": "post_42", "rank": 1, "score": 0.81, "reason_codes": ["TOPIC_MATCH"]}], "model_version": "content-v1"},
}
print(json.dumps(contracts, indent=2, ensure_ascii=False))

{
  "topic_request": {
    "post_id": "post_42",
    "title": "ESP32 robot",
    "body": "PID wall following",
    "image_ref": null
  },
  "topic_response": {
    "topics": [
      {
        "id": "Robotics",
        "score": 0.82
      }
    ],
    "model_version": "topic-v1"
  },
  "feedback_request": {
    "review_id": "review_7",
    "text": "Clear explanation but few examples",
    "stars": 4
  },
  "feedback_response": {
    "aspects": [
      {
        "name": "clarity",
        "sentiment": "positive"
      },
      {
        "name": "examples",
        "sentiment": "negative"
      }
    ],
    "text_signal": 0.1,
    "stars_unchanged": 4
  },
  "recommend_request": {
    "user_id": "user_8",
    "surface": "posts",
    "declared_topics": [
      "Programming/Web"
    ],
    "eligible_candidate_ids": [
      "post_42",
      "post_91"
    ]
  },
  "recommend_response": {
    "items": [
      {
        "id": "post_42",
        "rank": 1,
        "score": 0.81,
        "reason_

<h2 style="color:#0072B2;">1.6 Serialization Smoke Test</h2>

<span style="color:#D55E00;"><b>Teaching-only experiment:</b></span>  
The following tiny corpus tests the engineering path only. It is intentionally too small for evaluation, deployment, or claims about accuracy.

In [7]:
smoke_rows = [
    ("Build a responsive React dashboard with a Python API", ["Programming/Web"]),
    ("Learn JavaScript functions and CSS grid layouts", ["Programming/Web", "Design"]),
    ("Train a neural network for image classification", ["AI/Data"]),
    ("Analyze a dataset with pandas and machine learning", ["AI/Data", "Programming/Web"]),
    ("Program ESP32 sensors using interrupts and I2C", ["Electronics/Embedded", "Programming/Web"]),
    ("Design a low power embedded circuit with a microcontroller", ["Electronics/Embedded"]),
    ("Tune PID control for a line-following robot", ["Robotics", "Electronics/Embedded"]),
    ("Use computer vision to navigate an autonomous robot", ["Robotics", "AI/Data"]),
    ("Detect SQL injection and secure a web application", ["Cybersecurity", "Programming/Web"]),
    ("Study encryption, network attacks, and secure authentication", ["Cybersecurity"]),
    ("Create a mobile interface using typography and color theory", ["Design"]),
    ("Prototype an accessible user experience in Figma", ["Design"]),
    ("Solve differential equations and linear algebra exercises", ["Mathematics"]),
    ("Understand probability, vectors, and matrix multiplication", ["Mathematics", "AI/Data"]),
    ("Explain photosynthesis and cell biology experiments", ["Natural Sciences"]),
    ("Simulate electric fields using physics equations", ["Natural Sciences", "Mathematics"]),
]
smoke_df = pd.DataFrame(smoke_rows, columns=["text", "topics"])
smoke_df.head()

,text,topics
0,Build a responsive React dashboard with a Pyth...,[Programming/Web]
1,Learn JavaScript functions and CSS grid layouts,"[Programming/Web, Design]"
2,Train a neural network for image classification,[AI/Data]
3,Analyze a dataset with pandas and machine lear...,"[AI/Data, Programming/Web]"
4,Program ESP32 sensors using interrupts and I2C,"[Electronics/Embedded, Programming/Web]"


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer

label_encoder = MultiLabelBinarizer(classes=MVP_TOPICS)
Y = label_encoder.fit_transform(smoke_df["topics"])

topic_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
    ("classifier", OneVsRestClassifier(LogisticRegression(max_iter=1_000, random_state=SEED))),
])
topic_pipeline.fit(smoke_df["text"], Y)
print(f"Smoke pipeline fitted on {len(smoke_df)} illustrative rows. No metric is reported.")

Smoke pipeline fitted on 16 illustrative rows. No metric is reported.


In [9]:
def predict_topics(text, pipeline=topic_pipeline, encoder=label_encoder, threshold=0.35):
    probabilities = pipeline.predict_proba([text])[0]
    ranked = sorted(zip(encoder.classes_, probabilities), key=lambda pair: pair[1], reverse=True)
    selected = [{"topic": topic, "score": round(float(score), 4)} for topic, score in ranked if score >= threshold]
    if not selected:
        topic, score = ranked[0]
        selected = [{"topic": topic, "score": round(float(score), 4)}]
    return selected

KNOWN_TEXT = "Build an ESP32 robot that uses computer vision for navigation"
prediction_before_save = predict_topics(KNOWN_TEXT)
prediction_before_save

[{'topic': 'AI/Data', 'score': 0.291}]

<h3 style="color:#009E73;">Why Save Preprocessing with the Model?</h3>

The fitted TF-IDF vocabulary is part of the model contract. Rebuilding it later can map the same sentence to different features. We therefore serialize the entire pipeline, label order, threshold, taxonomy, and metadata together.

In [10]:
bundle = {
    "pipeline": topic_pipeline,
    "label_encoder": label_encoder,
    "threshold": 0.35,
    "taxonomy": MVP_TOPICS,
    "seed": SEED,
    "model_version": "topic-smoke-v0",
    "status": "SERIALIZATION_SMOKE_TEST_ONLY",
}

with tempfile.TemporaryDirectory() as temporary_directory:
    artifact_path = Path(temporary_directory) / "topic_smoke_bundle.joblib"
    joblib.dump(bundle, artifact_path)
    loaded_bundle = joblib.load(artifact_path)
    prediction_after_load = predict_topics(
        KNOWN_TEXT,
        pipeline=loaded_bundle["pipeline"],
        encoder=loaded_bundle["label_encoder"],
        threshold=loaded_bundle["threshold"],
    )
    saved_bytes = artifact_path.stat().st_size

assert prediction_before_save == prediction_after_load
print(f"Round-trip passed: {saved_bytes:,} bytes; prediction reproduced exactly.")
prediction_after_load

Round-trip passed: 21,994 bytes; prediction reproduced exactly.


[{'topic': 'AI/Data', 'score': 0.291}]

<h2 style="color:#0072B2;">1.7 Reproducibility Checklist</h2>

A real artifact is ready only when it includes: model and preprocessing, label order, threshold, seed, library versions, dataset/split identifiers, metrics, model version, and a known prediction test.

<span style="color:#D55E00;"><b>Security rule:</b></span> Never load an untrusted `joblib` file.

In [11]:
day1_status = pd.DataFrame([
    {"Item": "Sprint goal and three system boundaries", "Status": "Complete"},
    {"Item": "Eight-topic multi-label taxonomy", "Status": "Draft — team confirmation required"},
    {"Item": "Dataset/model mapping", "Status": "Candidate plan — audit required"},
    {"Item": "Reproducibility seed and environment capture", "Status": "Complete"},
    {"Item": "Save/load prediction smoke test", "Status": "Passed"},
    {"Item": "Real project model training and evaluation", "Status": "Not started — no claim made"},
])
day1_status

,Item,Status
0,Sprint goal and three system boundaries,Complete
1,Eight-topic multi-label taxonomy,Draft — team confirmation required
2,Dataset/model mapping,Candidate plan — audit required
3,Reproducibility seed and environment capture,Complete
4,Save/load prediction smoke test,Passed
5,Real project model training and evaluation,Not started — no claim made


<h2 style="color:#0072B2;">Day 1 Conclusion</h2>

We converted the research proposal into an explicit Sprint goal, backlog, taxonomy, model/data map, and service contracts. We also proved that a complete preprocessing-plus-model bundle can be saved and loaded without changing a known prediction.

<span style="color:#009E73;"><b>Day 2 handoff:</b></span>  
The smoke corpus is illustrative and produces no valid accuracy/F1 result. The next implementation step is to audit and adopt permitted real data, replace the smoke pipeline with an evaluated baseline, and expose only versioned artifacts through FastAPI.

### Tools Used

Scikit-learn • Joblib • Pandas • NumPy • Jupyter/Colab